# Mathematical model descriptions

`Model.to_latex()` describes the declared model without building, fitting or sampling it. The returned string is suitable for MathJax or `IPython.display.Math`. It includes response laws, parameter conversions, predictors, links and priors. Default text representations and inference remain unchanged.

The examples below are deterministic declarations. Rendering does not execute user-defined distributions, links or potentials. Such functions are named explicitly rather than assigned invented mathematical meanings. The examples cover linear and distributional model specifications; unsupported nonlinear component structures raise an error.


In [ ]:
import bambi as bmb
import numpy as np
import pandas as pd
from IPython.display import Math, display

frame = pd.DataFrame({
    "y": [0.4, 0.7, 1.0, 1.2, 0.8, 1.5],
    "x": [-1.0, -0.6, -0.2, 0.2, 0.6, 1.0],
    "group": ["a", "a", "b", "b", "c", "c"],
    "binary": [0, 1, 0, 1, 1, 0],
    "category": pd.Categorical(["a", "b", "c", "a", "c", "b"]),
    "weight": [1.0, 0.5, 1.0, 2.0, 1.0, 1.5],
})


## A Gaussian regression

The common design is centered when an intercept is present and `center_predictors=True`. Fixed auxiliary values, numeric precision and the prior hierarchy remain visible.


In [ ]:
gaussian = bmb.Model("y ~ x", frame, priors={"sigma": 0.8})
display(Math(gaussian.to_latex()))
assert not gaussian.built


## Varying effects and offsets

Group membership and varying coefficients are separate from the common design. An offset contributes its observed values rather than a sampled coefficient.


In [ ]:
hierarchical = bmb.Model("y ~ x + offset(x) + (1|group)", frame)
display(Math(hierarchical.to_latex()))
assert not hierarchical.built


## Distributional regression

Each modeled likelihood parameter has its own predictor and inverse link. Here the standard deviation uses a log link.


In [ ]:
distributional = bmb.Model(bmb.Formula("y ~ x", "sigma ~ x"), frame)
display(Math(distributional.to_latex()))
assert not distributional.built


## A binary response and success level

The description records the Bernoulli success level and the inverse-logit link, including the numerical endpoint convention used by the native link.


In [ ]:
binary = bmb.Model("binary ~ x", frame, family="bernoulli")
display(Math(binary.to_latex()))
assert not binary.built


## A categorical response

The first response category is the reference category. The displayed reference-softmax equations include all categories and preserve parameter aliases.


In [ ]:
categorical = bmb.Model("category ~ x", frame, family="categorical")
categorical.set_alias({"p": "chance"})
display(Math(categorical.to_latex()))
assert not categorical.built


## A weighted likelihood

Weights multiply log-likelihood contributions. They do not turn the response into a new probability distribution.


In [ ]:
weighted = bmb.Model("weighted(y, weight) ~ x", frame, priors={"sigma": 0.8})
display(Math(weighted.to_latex()))
assert not weighted.built


## A Hilbert-space Gaussian process

The mathematical description includes basis frequencies, spectral density and weight parameterization. Rendering does not construct the PyMC HSGP or sample its weights.


In [ ]:
smooth = bmb.Model("y ~ 0 + hsgp(x, m=4, c=1.5)", frame, priors={"sigma": 0.8})
smooth_term = next(iter(smooth.parameters["mu"].hsgp_terms.values()))
smooth.set_priors({smooth_term.name: {"sigma": 1.0, "ell": 0.7}})
display(Math(smooth.to_latex()))
assert not smooth.built


## Scope and limitations

The renderer supports the native families, prior hierarchies, group effects, offsets and HSGP specification on current `main`. It includes truncation, constraints and censoring boundaries; custom callable mathematics remain explicitly user defined. Unknown model structures raise a clear error rather than disappearing from the description.

The text reports the declared parameterization, not posterior inference or a causal interpretation. Existing fitted results, random generators and backend state are not used to construct it.
